# OMNI-917 · The event-only PFM trained again, its epoch chosen within each window (Colab GPU)

OMNI-916's event-only model had its epoch chosen by the pooled AUC, which pools every sample of every window (a
sample 30 s before the event counts as "event"; different places are compared). Here it is **trained again** with
every setting as OMNI-916 (D-62: `minus_first`, `labels="event"`, both domains, the same samples, splits and `α`,
3 seeds, up to 120 epochs, patience 30), except the epoch: after every epoch `train_pfm.validation` measures both
domains and **UCF-Crime's validation within-window AUC picks `best.pt`** (D-64, amended before training). Every
epoch's five measures go into the history.

**Which model the product serves (D-64):** the seed with the highest within-window AUC, if its **flat-line check**
on UCF-Crime's validation normal windows is **≤ 0.60**. If it fails, v2 stays served and nothing new is. v2's old
checkpoint (`pfm-joint-v2-mf-s2-e55`) is measured on the same validation for reference only; it chooses nothing.

1. **Setup:** clone, Drive
2. **The vectors:** as OMNI-905
3. **Training:** checkpoints in `models/pfm/joint-event-ww/seed*/` (new; the old folders untouched); if the session
   ends, Run all again
4. **The choice, v2 beside it, the rule and the record:** `models/pfm/runs/pfm-joint-v4-ev-ww-s{seed}-e{epoch}/`,
   `model_runs.json`, `results/omni917_joint.json`

Then, only if the rule is met, `notebooks/pfm_heldout_gpu.ipynb` with this run's id. Held-out data is not touched
here. Runtime → Change runtime type → **GPU**.

In [ ]:
# 1 · Setup
BRANCH = "slice-09-10-forecast"
REPO = "https://github.com/OmniSight-team/omnisight.git"

import os, subprocess, sys
if not os.path.isdir("/content/omni-sight"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "/content/omni-sight"], check=True)
else:
    subprocess.run(["git", "-C", "/content/omni-sight", "pull", "-q"], check=True)
COMMIT = subprocess.run(["git", "-C", "/content/omni-sight", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print(COMMIT)
sys.path.insert(0, "/content/omni-sight/scripts")

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import torch
assert torch.cuda.is_available(), "no GPU: Runtime -> Change runtime type -> GPU"
DEVICE, GPU = "cuda", torch.cuda.get_device_name(0)
print(torch.__version__, GPU)

In [ ]:
# 2 · Both domains' InternVideo2 vectors, copied from Drive once, and the samples
import hashlib, json, os, shutil, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import pfm_runs, pfm_samples, train_pfm
from pfm_model import Config

REPO_DIR = Path("/content/omni-sight")
DRIVE = Path("/content/drive/MyDrive/omnisight")
LOCAL = Path("/content/pfm")
DOMAINS = ("ccd", "ucf_crime")
SAMPLE_FILES = {d: REPO_DIR / "datasets" / "samples" / f"{d}.csv" for d in DOMAINS}
SAMPLES = {d: pfm_samples.read_samples(SAMPLE_FILES[d]) for d in DOMAINS}

def copy_folder(domain):
    keys = sorted({s["clip_key"] for s in SAMPLES[domain]})
    source, target = DRIVE / "embeddings" / "pfm" / domain / "internvideo2", LOCAL / domain
    def one(key):
        dst = target / f"{key}.npz"
        if not dst.is_file():  # copied under a temporary name, renamed when complete: a cut copy leaves no broken file
            dst.parent.mkdir(parents=True, exist_ok=True)
            part = dst.with_name(dst.name + ".part")
            shutil.copyfile(source / f"{key}.npz", part)
            os.replace(part, dst)
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(one, keys))
    return target

STORES, TRAIN, VAL = {}, {}, {}
for d in DOMAINS:
    start = time.time()
    STORES[d] = pfm_samples.Store(SAMPLES[d], copy_folder(d))
    TRAIN[d] = train_pfm.Data.split(STORES[d], SAMPLES[d], "train", d)
    VAL[d] = train_pfm.Data.split(STORES[d], SAMPLES[d], "validation", d)
    print(f"{d}: {len(STORES[d].vectors):,} vectors, train {len(TRAIN[d]):,} samples, validation {len(VAL[d]):,} "
          f"({time.time() - start:.0f} s)")

In [ ]:
# 3 · Training: event or normal (D-62), the scene removed (D-61), the epoch by the within-window AUC (D-64)
SEEDS = (0, 1, 2)
VAL_CLASSES = {d: VAL[d].for_labels("classes") for d in DOMAINS}  # each sample's own class, for v2's reference
TRAIN = {d: TRAIN[d].for_labels("event") for d in DOMAINS}  # D-62: every event class is one "event"
VAL = {d: VAL[d].for_labels("event") for d in DOMAINS}
ALPHAS = {d: float(f"{1.0 / np.nanvar(TRAIN[d].tau):.3g}") for d in DOMAINS}  # D-59, as OMNI-916
CONFIG = Config(input_dim=512, domains=list(DOMAINS), alphas=ALPHAS, input="minus_first", epochs=120, labels="event")
PATIENCE = 30
print("alphas", ALPHAS)
RUNS_DIR = DRIVE / "models" / "pfm" / "joint-event-ww"

def validate(model):  # D-64: score = UCF-Crime's within-window AUC; all five measures of both domains in the history
    return train_pfm.validation(model, VAL, DEVICE, choose_by="ucf_crime")

HISTORY = {}
for seed in SEEDS:
    print(f"seed {seed}")
    _, HISTORY[seed] = train_pfm.train(CONFIG, dict(TRAIN), RUNS_DIR / f"seed{seed}", seed=seed, device=DEVICE,
                                       validate=validate, patience=PATIENCE)

In [ ]:
# 4 · The best seed by the within-window AUC, v2 beside it, D-64's flat-line check, the record
best = {seed: max(HISTORY[seed], key=lambda h: (h["validation"]["score"], -h["epoch"])) for seed in SEEDS}
SEED = max(SEEDS, key=lambda s: (best[s]["validation"]["score"], -s))
model, state = train_pfm.load(RUNS_DIR / f"seed{SEED}" / "best.pt", DEVICE)
assert state["epoch"] == best[SEED]["epoch"]
VALIDATION = train_pfm.validation(model, VAL, DEVICE)
assert abs(VALIDATION["score"] - best[SEED]["validation"]["score"]) < 1e-6  # the reloaded model is the chosen one

V2 = pfm_runs.download(DRIVE, "pfm-joint-v2-mf-s2-e55", target=LOCAL / "v2")
v2_model, _ = train_pfm.load(LOCAL / "v2" / pfm_runs.CHECKPOINT, DEVICE)
V2_VALIDATION = train_pfm.validation(v2_model, VAL_CLASSES, DEVICE)  # reference only

def f(v, spec=".4f"):
    return "-" if v is None else format(v, spec)

ROWS = {f"event s{s} e{best[s]['epoch']}": best[s]["validation"] for s in SEEDS}
ROWS["v2 (reference)"] = V2_VALIDATION
for d in DOMAINS:
    print(f"\n{d} (validation)")
    print("  model               within (windows/skipped)  flat-line  per-window  AP      mTTA     detection  pooled  "
          "false alarms  θ")
    for key, v in ROWS.items():
        m = v[d]
        print(f"  {key:19s} {f(m['within_window_auc'])} ({m['within_window_windows']}/{m['within_window_skipped']})"
              f"{f(m['flat_line']):>17s}  {f(m['per_window_auc']):>10s}  {f(m['ap'])}  {f(m['mtta_s'], '.2f'):>5s} s  "
              f"{f(m['detection_rate'], '.3f'):>9s}  {f(m['pooled_auc'])}  {m['false_alarm_windows']:.3f}         "
              f"{m['theta']:.4f}")
for s in SEEDS:
    print(f"seed {s}: best epoch {best[s]['epoch']} of {len(HISTORY[s])} trained, last epoch's within-window "
          f"{HISTORY[s][-1]['validation']['score']:.4f}")

FLAT = VALIDATION["ucf_crime"]["flat_line"]
PASSES = FLAT is not None and FLAT <= 0.60
print(f"\nD-64: seed {SEED} (within-window AUC {VALIDATION['score']:.4f}), flat-line {f(FLAT)} (≤ 0.60) -> "
      f"{'served' if PASSES else 'fails the flat-line check: v2 stays served'}")

RUN_ID = f"pfm-joint-v4-ev-ww-s{SEED}-e{state['epoch']}"
folder = DRIVE / pfm_runs.RUNS / RUN_ID
folder.mkdir(parents=True, exist_ok=True)
shutil.copyfile(RUNS_DIR / f"seed{SEED}" / "best.pt", folder / pfm_runs.CHECKPOINT)
run = pfm_runs.record(DRIVE, RUN_ID, config=CONFIG.to_dict(), epochs_trained=len(HISTORY[SEED]), epoch=state["epoch"],
                      seed=SEED, thresholds={d: VALIDATION[d]["theta"] for d in DOMAINS},
                      validation={**VALIDATION, "flat_line_ucf_crime": FLAT, "d64_rule_met": PASSES},
                      commit=COMMIT, gpu=GPU, torch=torch.__version__, encoder="internvideo2",
                      false_alarm_rule=train_pfm.FALSE_ALARM,
                      samples_sha256={d: hashlib.sha256(SAMPLE_FILES[d].read_bytes()).hexdigest() for d in DOMAINS},
                      created_at=time.strftime("%Y-%m-%d %H:%M"))
pfm_runs.register(DRIVE, run, chosen=PASSES)
if not PASSES:  # v2 stays (or becomes again) the chosen run
    pfm_runs.register(DRIVE, V2, chosen=True)
print(RUN_ID, run["thresholds"], "chosen" if PASSES else "recorded, not chosen")

out = DRIVE / "results" / "omni917_joint.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps({"commit": COMMIT, "gpu": GPU, "torch": torch.__version__, "config": CONFIG.to_dict(),
    "seeds": SEEDS, "chosen": {"run_id": RUN_ID, "seed": SEED, "epoch": state["epoch"]},
    "best_per_seed": {s: {"epoch": h["epoch"], "score": h["validation"]["score"]} for s, h in best.items()},
    "history": HISTORY, "train_seconds": {s: sum(h["seconds"] for h in HISTORY[s]) for s in SEEDS},
    "validation": VALIDATION, "v2_validation": V2_VALIDATION, "flat_line_ucf_crime": FLAT, "d64_rule_met": PASSES},
    indent=1, default=float))
print(out)